# Calculate extremes in SMYLE FOSI

In [ ]:
# general use packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# packages for altering time to match up!
import sys
import cftime

# climpred packages
import climpred
from climpred import HindcastEnsemble
from climpred.tutorial import load_dataset
from climpred.stats import rm_poly

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat

In [ ]:
# var = 'photoC_TOT_zint_100m' # photoC_TOT_zint_100m, diazC diatC spC, diazChl diatChl spChl
thold = 0.1

In [ ]:
%%time
# # # FOSI
# fosi = xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.' + var + '.030601-036812.nc')[var]

fosi_1 = xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.diazC.030601-036812.nc')['diazC']
fosi_1 = fosi_1.isel(z_t_150m = slice(0,10)).sum('z_t_150m')
fosi_2 = xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.diatC.030601-036812.nc')['diatC']
fosi_2 = fosi_2.isel(z_t_150m = slice(0,10)).sum('z_t_150m')
fosi_3 = xr.open_dataset('/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.spC.030601-036812.nc')['spC']
fosi_3 = fosi_3.isel(z_t_150m = slice(0,10)).sum('z_t_150m')
fosi = fosi_1 + fosi_2 + fosi_3

In [ ]:
fosi = fosi_1 + fosi_2 + fosi_3

In [ ]:
fosi['time'] = pd.date_range("1958-01", "2020-12", freq="MS")

In [ ]:
ds = fosi # , fosi

ds = ds.drop('ULONG')
ds = ds.drop('ULAT')

In [ ]:
ds.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/LR/totC_zint_100m.nc')

## Regrid

In [ ]:
import xesmf as xe

from platform import python_version
print(xe.__version__)

In [ ]:
obs = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/OceanSODA-ETHZ_GRaCER_v2021a_1982-2020.nc')

In [ ]:
regridder_smyle = xe.Regridder(ds, obs, 'bilinear', periodic=True)

In [ ]:
regridder_smyle

In [ ]:
%%time
ds_rg = regridder_smyle(ds)

In [ ]:
# size of the dataset
print(ds_rg.nbytes / 1e9) # GB
print(ds.nbytes / 1e9) # GB

In [ ]:
var = 'totC_zint_100m'
ds_rg = ds_rg.to_dataset(name = var)

In [ ]:
# ds_rg.to_netcdf('/glade/work/smogen/extremes_Chl_Nikki_figures/data/'+ var +'.monthly.surface.regrid.log.nc')
ds_rg.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/FOSI/' + var +  '.monthly.no_log.regrid.nc', mode='w')
